In [ ]:
import torch  

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

In [ ]:
from pathlib import Path

yaml_files = list(Path("/kaggle/input").rglob("data.yaml"))

for f in yaml_files:
    print(f)

DATASET_DIR = yaml_files[0].parent
print("\nDATASET_DIR =", DATASET_DIR)

In [ ]:
for split in ["train", "valid", "test"]:
    images = DATASET_DIR / split / "images"
    labels = DATASET_DIR / split / "labels"

    print(f"\n{split}")
    print("Images folder:", images.exists())
    print("Labels folder:", labels.exists())

    if images.exists():
        print("Images:", len(list(images.iterdir())))

    if labels.exists():
        print("Labels:", len(list(labels.iterdir())))

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import random

# Get training images and labels
train_images = DATASET_DIR / "train" / "images"
train_labels = DATASET_DIR / "train" / "labels"

# Pick 5 random images
image_files = list(train_images.glob("*"))
sample_images = random.sample(image_files, 5)

for image_path in sample_images:

    # Corresponding label file
    label_path = train_labels / (image_path.stem + ".txt")

    # Read image
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]

    # Read YOLO annotations
    if label_path.exists():
        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            class_id, x_center, y_center, box_width, box_height = map(float, parts)

            # Convert normalized YOLO coordinates to pixels
            x_center *= w
            y_center *= h
            box_width *= w
            box_height *= h

            x1 = int(x_center - box_width / 2)
            y1 = int(y_center - box_height / 2)
            x2 = int(x_center + box_width / 2)
            y2 = int(y_center + box_height / 2)

            # Draw bounding box
            cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)

            # Label
            cv2.putText(
                image,
                "Accident",
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 0, 0),
                2
            )

    # Display
    plt.figure(figsize=(10, 7))
    plt.imshow(image)
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()
    

In [ ]:
yaml_path = "/kaggle/working/accident_dataset.yaml"

with open(yaml_path, "w") as f:
    f.write(f"""
path: {DATASET_DIR}
train: train/images
val: valid/images
test: test/images

nc: 1
names:
  0: Accident
""")

print(open(yaml_path).read())

In [ ]:
!pip install -q ultralytics


In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO model
model = YOLO("yolo11n.pt")

print("YOLO model loaded successfully!")


In [ ]:
results = model.train(
    data='/kaggle/working/accident_dataset.yaml',
    epochs=10,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project="/kaggle/working/runs",
    name="accident_detector"
)

In [ ]:
from ultralytics import YOLO

best_model_path = "/kaggle/working/runs/accident_detector-3/weights/best.pt"

model = YOLO(best_model_path)

print("Best trained model loaded successfully!")

In [ ]:
test_metrics = model.val(
    data=yaml_path,
    split="test",
    imgsz=640,
    device=0
)

In [ ]:
from pathlib import Path
import random
import matplotlib.pyplot as plt
import cv2

# Get test images
test_images = list((DATASET_DIR / "test" / "images").glob("*"))

# Select one random image
test_image = random.choice(test_images)

print("Testing image:", test_image.name)

# Run prediction
results = model.predict(
    source=str(test_image),
    conf=0.25,
    imgsz=640,
    device=0
)

# Draw predicted bounding box
predicted_image = results[0].plot()

# Convert image format for display
predicted_image = cv2.cvtColor(
    predicted_image,
    cv2.COLOR_BGR2RGB
)

# Display
plt.figure(figsize=(10, 7))
plt.imshow(predicted_image)
plt.title("Accident Detection Prediction")
plt.axis("off")
plt.show()

In [ ]:
from pathlib import Path
from IPython.display import display, Image

test_images = list((DATASET_DIR / "test" / "images").glob("*.jpg"))

for image_path in test_images[:5]:

    results = model.predict(
        source=str(image_path),
        conf=0.25,
        save=True
    )

    print("Image:", image_path.name)

    saved_image = Path(results[0].save_dir) / image_path.name

    display(Image(filename=str(saved_image)))